# Notebook for assigning lava bedding orientations to NSVG pmag sites

In [1]:
import geopandas as gpd
import folium
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, LineString
from geopy.distance import geodesic
from pyhigh import get_elevation
import numpy as np
import pmagpy.ipmag as ipmag
from strat_height import *

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## Make handy functions

In [2]:
def estimate_height_from_lava_base(folium_map, site_data, lava_polygon, datum,
                                   distance=5000, lat_col='lat', lon_col='lon', 
                                   dip_col='bedding_dip', dip_dir_col='bedding_dip_dir', 
                                   color='purple', multipoint_selection='min'):
    '''
    function for estimating the strat height of the site from the lava base

    Parameters
    ----------
    folium_map : folium map object
        folium map object
    site_data : pandas DataFrame
        site data
    lava_polygon : shapely Polygon object
        lava polygon
    datum : float
        datum for the calculated strat height
        need to assign to be the basis of a lava flow as provided by Green, 2011
    distance : float
        distance in meters to draw the line from the site toward the lava base
        for calculating the intersection point
    lat_col : str
        column name for latitude in site_data
    lon_col : str
        column name for longitude in site_data
    dip_col : str
        column name for dip in site_data
    dip_dir_col : str
        column name for dip direction in site_data
    color : str
        color of the circle marker
        
    '''
    for i, r in site_data.iterrows():
        site_elevation = get_elevation(r[lat_col], r[lon_col]-360)
        line = create_line_from_point(r[lon_col], r[lat_col], r[dip_dir_col]+180, distance=distance)
        intersection = line.intersection(lava_polygon.exterior)
        if intersection.type == 'Point':
            folium.CircleMarker([intersection.y, intersection.x], 
                                radius=5,  # Size of the circle marker
                                color="",  # Border color of the circle
                                fill=True,
                                fill_color=color,  # Fill color of the circle
                                fill_opacity=0.7).add_to(folium_map)
        elif intersection.type == 'MultiPoint':
            # there are multiple points, we want the one with smallest longitude in the MultiPoint object
            if multipoint_selection == 'min':
                intersection = min(intersection.geoms, key=lambda point: point.x)
            elif multipoint_selection == 'max':
                intersection = max(intersection.geoms, key=lambda point: point.x)
            folium.CircleMarker([intersection.y, intersection.x], 
                                radius=5,  # Size of the circle marker
                                color="",  # Border color of the circle
                                fill=True,
                                fill_color=color,  # Fill color of the circle
                                fill_opacity=0.7).add_to(folium_map)
        # print('site lat', r[lat_col], 'site lon', r[lon_col], 'site elevation', site_elevation, 'intersection lat', intersection.y, 'intersection lon', intersection.x)
        
        intersection_elevation = get_elevation(intersection.y, intersection.x-360)
        folium.PolyLine(locations=[(r[lat_col], r[lon_col]), (intersection.y, intersection.x)], color=color).add_to(folium_map)
        
        # now calculate the distance in meters of the line from the intersection point to the site
        distance_from_flowbase = geodesic((r[lat_col], r[lon_col]), (intersection.y, intersection.x)).meters

        # site_data.loc[i, 'distance_from_flowbase'] = distance_from_flowbase
        # write the distance from the flow base to the site in the original site data table
        if intersection_elevation < site_elevation:
            site_data.loc[i, 'relative height'] = (distance_from_flowbase + (site_elevation - intersection_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))
            site_data.loc[i, 'height'] = datum + (distance_from_flowbase + (site_elevation - intersection_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))
        else:
            site_data.loc[i, 'relative height'] = (distance_from_flowbase - np.abs(intersection_elevation - site_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))
            site_data.loc[i, 'height'] = datum + (distance_from_flowbase - np.abs(intersection_elevation - site_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))

## Load Green 2011 stratigraphy
- to get the base strat heights of the lava flows

In [3]:
NSVG_SW_strat = pd.read_csv('../../data/stratigraphy/NSVG_SW_strat.csv', index_col='Lava')
NSVG_SW_strat

,thickness,lava base strat height
Lava,,
Ely’s Peak basalts,370,0
Leif Erickson Park lavas,1160,370
Lakeside lavas,1285,1530
Lakewood lavas,1350,2815
Sucker River basalts,1500,4165
Larsmont basalts,550,5665
Two Harbors basalts,315,6215
Gooseberry River basalts,730,6530
Silver Bay porphyritic basalts,20,7260


In [4]:
NSVG_NE_strat = pd.read_csv('../../data/stratigraphy/NSVG_NE_strat.csv', index_col='Lava')
NSVG_NE_strat

,thickness,lava base strat height
Lava,,
Grand Portage lavas,945,0
Deronda Bay andesite,92,945
Red Rock rhyolite,67,1037
Hovland lavas,1932,1104
Brule River lavas,900,3036
Devil's Kettle rhyolite,235,3936
Naniboujou basalts,198,4171
Marr Island lavas,975,4369
porphyritic rhyolite,64,5344


## Load bedding data

- most of the NSVG orientation data are manually exported from the M119 bedrock geology map structural data 
- The Gooseberry bedding orientation is from Swanson-Hysell 2019 that were collected in pair with the pmag data from that study

In [5]:
Lakewood_bedding = pd.read_csv('../../data/GIS/M119/Lakewood_bedding.csv')
Sucker_River_bedding = pd.read_csv('../../data/GIS/M119/Sucker_River_bedding.csv')
Larsmont_TwoHarbor_bedding = pd.read_csv('../../data/GIS/M119/Larsmont_TwoHarbor_bedding.csv')
Schroeder_NE_bedding = pd.read_csv('../../data/GIS/M119/Schroeder_NE_bedding.csv')
Schroeder_middle_bedding = pd.read_csv('../../data/GIS/M119/Schroeder_middle_bedding.csv')
Schroeder_SW_bedding = pd.read_csv('../../data/GIS/M119/Schroeder_SW_bedding.csv')
Terrace_GoodHarbor_Breakwater_bedding = pd.read_csv('../../data/GIS/M119/Terrace_GoodHarbor_Breakwater_bedding.csv')
Croftville_bedding = pd.read_csv('../../data/GIS/M119/Croftville_bedding.csv')
Marr_island_bedding = pd.read_csv('../../data/GIS/M119/Marr_island_bedding.csv')
RedCliff_Kimball_bedding = pd.read_csv('../../data/GIS/M119/RedCliff_Kimball_bedding.csv')
RedRock_GrandPortage_bedding = pd.read_csv('../../data/GIS/M119/RedRock_GrandPortage_bedding.csv')

Gooseberry_bedding = pd.read_csv('../../data/pmag_compiled/Swanson-Hysell2019a/Gooseberry/ALL_plane.csv')

## Load pmag data

- Tauxe and Kodama 2009
- Swanson-Hysell et al., 2019
- Book 1968, 1972

### Tauxe and Kodama 2009

In [6]:
Tauxe2009_data = pd.read_csv('../../data/pmag_compiled/Tauxe2009/sites.txt', sep='\t', header=1)

Tauxe2009_Lakewood_site = ['ns053', 'ns055', 'ns057', 'ns060', 'ns061',
                   'ns062', 'ns063', 'ns064', 'ns065', 'ns066',
                   'ns067', 'ns068', 'ns071', 'ns072', 'ns073',
                   'ns074', 'ns075', 'ns077', 'ns078', 'ns079',
                   'ns080', 'ns081', 'ns083', 'ns085', 'ns087', 
                   'ns050', 'ns051', 'ns059']
Tauxe2009_Lakewood = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_Lakewood_site)]
Tauxe2009_Lakewood = assign_lava_bedding_to_sites(Tauxe2009_Lakewood, pd.concat([Sucker_River_bedding, Lakewood_bedding, Larsmont_TwoHarbor_bedding]))

Tauxe2009_SRB_site =  ['ns040', 'ns042', 'ns043', 'ns044', 'ns045',
                        'ns046', 'ns047', 'ns048', 'ns049', 'ns052', 
                        'ns054', 'ns056', 'ns058']
Tauxe2009_SRB = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_SRB_site)]
Tauxe2009_SRB = assign_lava_bedding_to_sites(Tauxe2009_SRB, pd.concat([Sucker_River_bedding, Lakewood_bedding, Larsmont_TwoHarbor_bedding]))

Tauxe2009_Larsmont_site = ['ns038', 'ns039']
Tauxe2009_Larsmont = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_Larsmont_site)]
Tauxe2009_Larsmont = assign_lava_bedding_to_sites(Tauxe2009_Larsmont, pd.concat([Sucker_River_bedding, Lakewood_bedding, Larsmont_TwoHarbor_bedding]))

Tauxe2009_TwoHarbors_site = ['ns034', 'ns035', 'ns036', 'ns037']
Tauxe2009_TwoHarbors= Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_TwoHarbors_site)]
Tauxe2009_TwoHarbors = assign_lava_bedding_to_sites(Tauxe2009_TwoHarbors, pd.concat([Sucker_River_bedding, Lakewood_bedding, Larsmont_TwoHarbor_bedding]))

Tauxe2009_Schroeder_SW_site = ['ns014', 'ns015']
Tauxe2009_Schroeder_SW = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_Schroeder_SW_site)]
Tauxe2009_Schroeder_SW = assign_lava_bedding_to_sites(Tauxe2009_Schroeder_SW, Schroeder_SW_bedding)

Tauxe2009_Schroeder_middle_site = ['ns008', 'ns009', 'ns010', 'ns011',
                                    'ns012', 'ns013']
Tauxe2009_Schroeder_middle = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_Schroeder_middle_site)]
Tauxe2009_Schroeder_middle = assign_lava_bedding_to_sites(Tauxe2009_Schroeder_middle, Schroeder_middle_bedding)

Tauxe2009_Schroeder_NE_site = ['ns006', 'ns007']
Tauxe2009_Schroeder_NE = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_Schroeder_NE_site)]
Tauxe2009_Schroeder_NE = assign_lava_bedding_to_sites(Tauxe2009_Schroeder_NE, Schroeder_NE_bedding)

Tauxe2009_TerracePoint_site = ['ns005']
Tauxe2009_TerracePoint = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_TerracePoint_site)]
Tauxe2009_TerracePoint = assign_lava_bedding_to_sites(Tauxe2009_TerracePoint, Terrace_GoodHarbor_Breakwater_bedding)

Tauxe2009_GoodHarbor_site = ['ns003', 'ns002', 'ns004', 'ns032']
Tauxe2009_GoodHarbor = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_GoodHarbor_site)]
Tauxe2009_GoodHarbor = assign_lava_bedding_to_sites(Tauxe2009_GoodHarbor, Terrace_GoodHarbor_Breakwater_bedding)

Tauxe2009_Breakwater_site = ['ns016', 'ns019']
Tauxe2009_Breakwater = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_Breakwater_site)]
Tauxe2009_Breakwater = assign_lava_bedding_to_sites(Tauxe2009_Breakwater, Terrace_GoodHarbor_Breakwater_bedding)

Tauxe2009_Croftville_site = ['ns031', 'ns028']
Tauxe2009_Croftville = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_Croftville_site)]
Tauxe2009_Croftville = assign_lava_bedding_to_sites(Tauxe2009_Croftville, Croftville_bedding)

Tauxe2009_RedCliff_Kimball_site = ['ns020', 'ns023', 'ns018', 'ns021']
Tauxe2009_RedCliff_Kimball = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_RedCliff_Kimball_site)]
Tauxe2009_RedCliff_Kimball = assign_lava_bedding_to_sites(Tauxe2009_RedCliff_Kimball, RedRock_GrandPortage_bedding)

Tauxe2009_MarrIsland_site = ['ns030', 'ns022']
Tauxe2009_MarrIsland = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_MarrIsland_site)]
Tauxe2009_MarrIsland = assign_lava_bedding_to_sites(Tauxe2009_MarrIsland, RedRock_GrandPortage_bedding)

Tauxe2009_GrandPortage_site = ['ns024', 'ns026']
Tauxe2009_GrandPortage = Tauxe2009_data[Tauxe2009_data.site.isin(Tauxe2009_GrandPortage_site)]
Tauxe2009_GrandPortage = assign_lava_bedding_to_sites(Tauxe2009_GrandPortage, RedRock_GrandPortage_bedding)

{'dec': 117.41874263768318, 'inc': 17.36274991318088, 'n': 23, 'r': 21.943306064797085, 'k': 20.81965199864176, 'alpha95': 6.795291640724391, 'csd': 17.752040739109823}
{'dec': 117.41874263768318, 'inc': 17.36274991318088, 'n': 23, 'r': 21.943306064797085, 'k': 20.81965199864176, 'alpha95': 6.795291640724391, 'csd': 17.752040739109823}
{'dec': 117.41874263768318, 'inc': 17.36274991318088, 'n': 23, 'r': 21.943306064797085, 'k': 20.81965199864176, 'alpha95': 6.795291640724391, 'csd': 17.752040739109823}
{'dec': 117.41874263768318, 'inc': 17.36274991318088, 'n': 23, 'r': 21.943306064797085, 'k': 20.81965199864176, 'alpha95': 6.795291640724391, 'csd': 17.752040739109823}
{'dec': 128.53051149843054, 'inc': 14.025128002680162, 'n': 2, 'r': 1.9952721748912134, 'k': 211.5137461708393, 'alpha95': 17.257863616490177, 'csd': 5.5694937416922}
{'dec': 136.90310173836264, 'inc': 14.822319906650362, 'n': 23, 'r': 22.146895835611204, 'k': 25.788175604279033, 'alpha95': 6.076844667216451, 'csd': 15.950

### Swanson-Hysell et al., 2019

In [7]:
SH2019_Gooseberry= pd.read_csv('../../data/pmag_compiled/Swanson-Hysell2019a/Gooseberry/sites.txt', sep='\t', header=1)
SH2019_strat_data = pd.read_csv('../../data/pmag_compiled/Swanson-Hysell2019a/Gooseberry/GB_strat_pmag.csv', index_col='Site')
# map the strat data to the site data
SH2019_Gooseberry['relative_strat'] = SH2019_Gooseberry['site'].map(SH2019_strat_data['STRAT_EC']) 
SH2019_Gooseberry['lon'] = SH2019_Gooseberry['lon'] % 360
SH2019_Gooseberry = assign_lava_bedding_to_sites(SH2019_Gooseberry, Gooseberry_bedding, 'dip_dir', 'dip')

{'dec': 114.82971259836229, 'inc': 11.040341503963417, 'n': 146, 'r': 139.95715533063645, 'k': 23.99532139807127, 'alpha95': 2.432808544273222, 'csd': 16.53566758830724}


### Books 1968, 1972

In [8]:
Books1968_data = pd.read_csv('../../data/pmag_compiled/Books1968/sites.txt', sep='\t', header=1)
Books1968_GrandPortage = Books1968_data[Books1968_data.location == 'Grand Portage']
Books1968_GrandPortage['lon'] = Books1968_GrandPortage['lon'] % 360
Books1968_GrandPortage = assign_lava_bedding_to_sites(Books1968_GrandPortage, RedRock_GrandPortage_bedding)

Books1972a_data = pd.read_csv('../../data/pmag_compiled/Books1972/sites.txt', sep='\t', header=1)
Books1972a_data['lon'] = Books1972a_data['lon'] % 360

Books1972a_MarrIsland_site = ['NS378','NS227', 'NS229']
Books1972a_MarrIsland = Books1972a_data[Books1972a_data.site.isin(Books1972a_MarrIsland_site)]
Books1972_MarrIsland = assign_lava_bedding_to_sites(Books1972a_MarrIsland, RedRock_GrandPortage_bedding)

Books1972a_Naniboujou_site = ['NS269']
Books1972a_Naniboujou = Books1972a_data[Books1972a_data.site.isin(Books1972a_Naniboujou_site)]
Books1972a_Naniboujou = assign_lava_bedding_to_sites(Books1972a_Naniboujou, RedRock_GrandPortage_bedding)

Books1972a_RedCliff_Kimball_site = ['NS226', 'NS375']
Books1972a_RedCliff_Kimball = Books1972a_data[Books1972a_data.site.isin(Books1972a_RedCliff_Kimball_site)]
Books1972a_RedCliff_Kimball = assign_lava_bedding_to_sites(Books1972a_RedCliff_Kimball, RedRock_GrandPortage_bedding)

Books1972a_Croftville_site = ['NS362','NS365']
Books1972a_Croftville = Books1972a_data[Books1972a_data.site.isin(Books1972a_Croftville_site)]
Books1972a_Croftville = assign_lava_bedding_to_sites(Books1972a_Croftville, Croftville_bedding)

Books1972a_GoodHarbor_site = ['NS367','NS265','NS368', 'NS369', 'NS374', 'NS376',
                                'NS377', 'NS169', 'NS170', 'NS171']
Books1972a_GoodHarbor = Books1972a_data[Books1972a_data.site.isin(Books1972a_GoodHarbor_site)]
Books1972a_GoodHarbor = assign_lava_bedding_to_sites(Books1972a_GoodHarbor, Terrace_GoodHarbor_Breakwater_bedding)

Books1972a_TerracePoint_site = ['NS370', 'NS371', 'NS372']
Books1972a_TerracePoint = Books1972a_data[Books1972a_data.site.isin(Books1972a_TerracePoint_site)]
Books1972a_TerracePoint = assign_lava_bedding_to_sites(Books1972a_TerracePoint, Schroeder_middle_bedding)

Books1972a_Schroeder_middle_site = ['NS263','NS256', 'NS255', 'NS254', 'NS379',
                                    'NS380', 'NS381', 'NS382']
Books1972a_Schroeder_middle = Books1972a_data[Books1972a_data.site.isin(Books1972a_Schroeder_middle_site)]
Books1972a_Schroeder_middle = assign_lava_bedding_to_sites(Books1972a_Schroeder_middle, Schroeder_middle_bedding)

Books1972a_Schroeder_upper_site = ['NS264', 'NS257', 'NS258', 'NS259',
                                    'NS260', 'NS261', 'NS262']
Books1972a_Schroeder_upper = Books1972a_data[Books1972a_data.site.isin(Books1972a_Schroeder_upper_site)]
Books1972a_Schroeder_upper = assign_lava_bedding_to_sites(Books1972a_Schroeder_upper, Schroeder_NE_bedding)

{'dec': 196.25644489235546, 'inc': 10.797649376089552, 'n': 4, 'r': 3.9815414921027386, 'k': 162.5266796589278, 'alpha95': 7.2286414543008055, 'csd': 6.353641221481646}
{'dec': 196.25644489235546, 'inc': 10.797649376089552, 'n': 4, 'r': 3.9815414921027386, 'k': 162.5266796589278, 'alpha95': 7.2286414543008055, 'csd': 6.353641221481646}
{'dec': 196.25644489235546, 'inc': 10.797649376089552, 'n': 4, 'r': 3.9815414921027386, 'k': 162.5266796589278, 'alpha95': 7.2286414543008055, 'csd': 6.353641221481646}
{'dec': 196.25644489235546, 'inc': 10.797649376089552, 'n': 4, 'r': 3.9815414921027386, 'k': 162.5266796589278, 'alpha95': 7.2286414543008055, 'csd': 6.353641221481646}
{'dec': 190.0, 'inc': 8.0}
{'dec': 175.0, 'inc': 15.0}
{'dec': 136.90310173836264, 'inc': 14.822319906650362, 'n': 23, 'r': 22.146895835611204, 'k': 25.788175604279033, 'alpha95': 6.076844667216451, 'csd': 15.95051498535234}
{'dec': 136.90310173836264, 'inc': 14.822319906650362, 'n': 23, 'r': 22.146895835611204, 'k': 25.78

## load NSVG polygons

In [9]:
NSVG_polygons = gpd.read_file("../../data/GIS/M119/NSVG_polygons.shp")
NSVG_polygons = NSVG_polygons.to_crs(epsg=4326)
# make sure all the polygons longitudes are between 0 and 360
NSVG_polygons = NSVG_polygons.explode(ignore_index=True)
NSVG_polygons['geometry'] = [Polygon([(x[0] % 360, x[1]) for x in poly.exterior.coords]) for poly in NSVG_polygons['geometry']]

NSVG_EPB = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Ely')].unary_union

NSVG_LEPB = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Leif')].unary_union

NSVG_Lakeside = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Lakeside')].unary_union

NSVG_Lakewood = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Lakewood')].unary_union

NSVG_SRB = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Sucker')].unary_union

NSVG_Larsmont = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Larsmont')].unary_union

NSVG_TwoHarbors = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Two Harbors')].unary_union

NSVG_Gooseberry = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Gooseberry')].unary_union

NSVG_SBPB = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Silver Bay')].unary_union

NSVG_PR = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Palisade')].unary_union

NSVG_Schroeder = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Schoeder')].unary_union

NSVGn = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('NSVGn - ophitic basalt')].unary_union

NSVG_GrandPortage = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Grand Portage')].unary_union

NSVG_MarrIsland = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Marr Island')].unary_union

NSVG_Naniboujou = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Naniboujou')].unary_union

NSVG_Kimball = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Kimball')].unary_union

NSVG_RedCliff = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Red Cliff')].unary_union

NSVG_Croftville = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Croftville')].unary_union

NSVG_Breakwater = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Breakwater')].unary_union

NSVG_GoodHarbor = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Good Harbor')].unary_union

NSVG_TerracePoint = NSVG_polygons[NSVG_polygons['DESCRIPTN'].str.contains('Terrace Point')].unary_union

## Make the map

In [11]:
# plot the sites
NSVG_map = folium.Map(location=[Tauxe2009_data['lat'].mean(), Tauxe2009_data['lon'].mean()], 
                      zoom_start=9, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")

plot_lava(NSVG_map, NSVG_Lakewood, 'purple')
plot_site(NSVG_map, Tauxe2009_Lakewood, 'purple')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_Lakewood, NSVG_Lakewood, NSVG_SW_strat.loc['Lakewood lavas', 'lava base strat height'])

plot_lava(NSVG_map, NSVG_SRB, 'orange')
plot_site(NSVG_map, Tauxe2009_SRB, 'orange')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_SRB, NSVG_SRB, NSVG_SW_strat.loc['Sucker River basalts', 'lava base strat height'], color='orange', distance=10000, multipoint_selection='min')

plot_lava(NSVG_map, NSVG_Larsmont, 'yellow')
plot_site(NSVG_map, Tauxe2009_Larsmont, 'yellow')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_Larsmont, NSVG_SRB, NSVG_SW_strat.loc['Sucker River basalts', 'lava base strat height'], color='yellow', distance=10000)

plot_lava(NSVG_map, NSVG_TwoHarbors, 'pink')
plot_site(NSVG_map, Tauxe2009_TwoHarbors, 'pink')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_TwoHarbors, NSVG_TwoHarbors, NSVG_SW_strat.loc['Two Harbors basalts', 'lava base strat height'], color='pink', distance=10000)

plot_lava(NSVG_map, NSVG_Gooseberry, 'black')
plot_site(NSVG_map, SH2019_Gooseberry, 'black')
# For Gooseberry we only need to calcualte the height for the GB1 which is the lowest site, and add the value to the already measured strat heights in the field for the other sites
estimate_height_from_lava_base(NSVG_map, SH2019_Gooseberry, NSVG_Gooseberry.geoms[4], NSVG_SW_strat.loc['Gooseberry River basalts', 'lava base strat height'], color='black', distance=20000)
SH2019_Gooseberry_GB1 = SH2019_Gooseberry[SH2019_Gooseberry.site == 'GB1'].iloc[0]
SH2019_Gooseberry['height'] = SH2019_Gooseberry_GB1['height'] + SH2019_Gooseberry['relative_strat'] - SH2019_Gooseberry_GB1['relative_strat']

plot_lava(NSVG_map, NSVG_Schroeder, 'indigo')
plot_site(NSVG_map, Tauxe2009_Schroeder_SW, 'indigo')
plot_site(NSVG_map, Tauxe2009_Schroeder_middle, 'indigo')
plot_site(NSVG_map, Tauxe2009_Schroeder_NE, 'indigo')
plot_site(NSVG_map, Books1972a_Schroeder_middle, 'indigo')
plot_site(NSVG_map, Books1972a_Schroeder_upper, 'indigo')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_Schroeder_SW, NSVG_Schroeder.geoms[3], NSVG_SW_strat.loc['Schroeder Basalts', 'lava base strat height'], color='indigo', distance=10000)
estimate_height_from_lava_base(NSVG_map, Tauxe2009_Schroeder_middle, NSVG_Schroeder.geoms[3], NSVG_SW_strat.loc['Schroeder Basalts', 'lava base strat height'], color='indigo', distance=10000)
estimate_height_from_lava_base(NSVG_map, Tauxe2009_Schroeder_NE, NSVG_Schroeder.geoms[3], NSVG_NE_strat.loc['Lutsen basalts ', 'lava base strat height'], color='indigo', distance=10000)
estimate_height_from_lava_base(NSVG_map, Books1972a_Schroeder_middle, NSVG_Schroeder.geoms[3], NSVG_SW_strat.loc['Schroeder Basalts', 'lava base strat height'], color='indigo', distance=10000)
estimate_height_from_lava_base(NSVG_map, Books1972a_Schroeder_upper, NSVG_Schroeder.geoms[3], NSVG_NE_strat.loc['Lutsen basalts ', 'lava base strat height'], color='indigo', distance=10000)

plot_lava(NSVG_map, NSVG_TerracePoint, 'red')
plot_site(NSVG_map, Tauxe2009_TerracePoint, 'red')
plot_site(NSVG_map, Books1972a_TerracePoint, 'red')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_TerracePoint, NSVG_GoodHarbor, NSVG_NE_strat.loc['Good Harbor Bay andesite and Terrace Point basalt', 'lava base strat height'], color='red', distance=10000)
estimate_height_from_lava_base(NSVG_map, Books1972a_TerracePoint, NSVG_TerracePoint, NSVG_NE_strat.loc['Good Harbor Bay andesite and Terrace Point basalt', 'lava base strat height'], color='red', distance=10000)

plot_lava(NSVG_map, NSVG_GoodHarbor, 'blue')
plot_site(NSVG_map, Tauxe2009_GoodHarbor, 'blue')
plot_site(NSVG_map, Books1972a_GoodHarbor, 'blue')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_GoodHarbor, NSVG_GoodHarbor, NSVG_NE_strat.loc['Good Harbor Bay andesite and Terrace Point basalt', 'lava base strat height'], color='blue', distance=10000)
estimate_height_from_lava_base(NSVG_map, Books1972a_GoodHarbor, NSVG_GoodHarbor, NSVG_NE_strat.loc['Good Harbor Bay andesite and Terrace Point basalt', 'lava base strat height'], color='blue', distance=10000)

plot_lava(NSVG_map, NSVG_Breakwater, 'green')
plot_site(NSVG_map, Tauxe2009_Breakwater, 'green')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_Breakwater, NSVG_Breakwater.geoms[0], NSVG_NE_strat.loc['Breakwater basalt', 'lava base strat height'], color='green', distance=10000)

plot_lava(NSVG_map, NSVG_Croftville, 'brown')
plot_site(NSVG_map, Tauxe2009_Croftville, 'brown')
plot_site(NSVG_map, Books1972a_Croftville, 'brown')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_Croftville, NSVG_Croftville.geoms[1], NSVG_NE_strat.loc['Croftville basalts', 'lava base strat height'], color='brown', distance=10000)
estimate_height_from_lava_base(NSVG_map, Books1972a_Croftville, NSVG_Croftville.geoms[1], NSVG_NE_strat.loc['Croftville basalts', 'lava base strat height'], color='brown', distance=10000)

plot_lava(NSVG_map, NSVG_Kimball, 'gray')
plot_site(NSVG_map, Tauxe2009_RedCliff_Kimball, 'gray')
plot_site(NSVG_map, Books1972a_RedCliff_Kimball, 'gray')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_RedCliff_Kimball, NSVG_Kimball.geoms[1], NSVG_NE_strat.loc['Kimball Creek rhyolite', 'lava base strat height'], color='gray', distance=10000, multipoint_selection='max')
estimate_height_from_lava_base(NSVG_map, Books1972a_RedCliff_Kimball, NSVG_Kimball.geoms[1], NSVG_NE_strat.loc['Kimball Creek rhyolite', 'lava base strat height'], color='gray', distance=10000, multipoint_selection='max')

plot_lava(NSVG_map, NSVG_Naniboujou, 'dodgerblue')
plot_site(NSVG_map, Books1972a_Naniboujou, 'dodgerblue')
estimate_height_from_lava_base(NSVG_map, Books1972a_Naniboujou, NSVG_Naniboujou, NSVG_NE_strat.loc['Naniboujou basalts', 'lava base strat height'], color='dodgerblue', distance=20000, multipoint_selection='max')

plot_site(NSVG_map, Tauxe2009_MarrIsland, 'limegreen')
plot_site(NSVG_map, Books1972_MarrIsland, 'limegreen')
plot_lava(NSVG_map, NSVG_MarrIsland, 'limegreen')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_MarrIsland, NSVG_MarrIsland, NSVG_NE_strat.loc['Marr Island lavas', 'lava base strat height'], color='limegreen', distance=20000, multipoint_selection='max')
estimate_height_from_lava_base(NSVG_map, Books1972_MarrIsland, NSVG_MarrIsland, NSVG_NE_strat.loc['Marr Island lavas', 'lava base strat height'], color='limegreen', distance=20000, multipoint_selection='max')

plot_lava(NSVG_map, NSVG_GrandPortage, 'gold')
plot_site(NSVG_map, Tauxe2009_GrandPortage, 'gold')
plot_site(NSVG_map, Books1968_GrandPortage, 'gold')
estimate_height_from_lava_base(NSVG_map, Tauxe2009_GrandPortage, NSVG_GrandPortage.geoms[0], NSVG_NE_strat.loc['Grand Portage lavas', 'lava base strat height'], color='gold', distance=10000, multipoint_selection='max')
estimate_height_from_lava_base(NSVG_map, Books1968_GrandPortage, NSVG_GrandPortage.geoms[0], NSVG_NE_strat.loc['Grand Portage lavas', 'lava base strat height'], color='gold', distance=10000, multipoint_selection='max')  


# other NSVG lava flows that were not sampled
# plot_lava(NSVG_map, NSVG_EPB, 'red')
# plot_lava(NSVG_map, NSVG_LEPB, 'blue')
# plot_lava(NSVG_map, NSVG_Lakeside, 'green')
# plot_lava(NSVG_map, NSVG_SBPB, 'brown')
# plot_lava(NSVG_map, NSVG_PR, 'gray')
# plot_lava(NSVG_map, NSVGn, 'magenta')
NSVG_map

## Save data

In [ ]:
def save_data(file_name, site_data):
    # Write the extra row to the file first
    with open(file_name, 'w') as f:
        f.write('tab\tsite\n')  # Writing the extra row

    # Append the DataFrame to the file with the original headers
    site_data.to_csv(file_name, mode='a', index=False, sep='\t')

In [ ]:
Tauxe2009_with_height = pd.concat([Tauxe2009_Lakewood, Tauxe2009_SRB, Tauxe2009_Larsmont, 
                                   Tauxe2009_TwoHarbors, Tauxe2009_Schroeder_SW, Tauxe2009_Schroeder_middle, 
                                   Tauxe2009_Schroeder_NE, Tauxe2009_TerracePoint, Tauxe2009_GoodHarbor, 
                                   Tauxe2009_Breakwater, Tauxe2009_Croftville, Tauxe2009_RedCliff_Kimball, 
                                   Tauxe2009_MarrIsland, Tauxe2009_GrandPortage])

Books1972_with_height = pd.concat([Books1972a_MarrIsland, Books1972a_Naniboujou, Books1972a_RedCliff_Kimball,
                                      Books1972a_Croftville, Books1972a_GoodHarbor, Books1972a_TerracePoint,
                                      Books1972a_Schroeder_middle, Books1972a_Schroeder_upper])

SH2019_with_height = SH2019_Gooseberry

In [ ]:
# save the site data
# Tauxe and Kodama 2009
TK09_file_name = '../../data/pmag_compiled/Tauxe2009/sites_with_height.txt'
Books1968_GrandPortage_file_name = '../../data/pmag_compiled/Books1968/GrandPortage_sites_with_height.txt'
Books1972_file_name = '../../data/pmag_compiled/Books1972/NSVG_sites_with_height.txt'
SH2019_file_name = '../../data/pmag_compiled/Swanson-Hysell2019a/Gooseberry/sites_with_height.txt'

save_data(TK09_file_name, Tauxe2009_with_height)
save_data(Books1968_GrandPortage_file_name, Books1968_GrandPortage)
save_data(Books1972_file_name, Books1972_with_height)
save_data(SH2019_file_name, SH2019_with_height)